# IFRS S1/S2 Writer Stage

Consumes `generation_blocks_<bank>.json` + `evidence_store_<bank>.json` + `section_plan_<bank>.json`
and produces the aligned report. Per block: build prompt → call Azure → parse the JSON contract →
**deterministic gate** (re-resolve every cited `citation_id` to the store and check the number matches)
→ bounded revision on failure → assemble into section > subsection > block.

Traceability is enforced by code, not trusted: a hallucinated or wrong number cannot pass the gate.

In [ ]:
import os, re, json, time
from pathlib import Path
from collections import defaultdict

try:
    import requests
except ImportError:
    requests = None

OUT = Path("mapping_outputs")
BANK = "BANK01"
MOCK_MODE = False
JSON_MODE = True
TEMPERATURE = 0.2
MAX_TOKENS = 8000          # reasoning/large blocks: keep high to avoid truncated JSON
MAX_REVISIONS = 2


# ---------------------------------------------------------------------------
# Azure fast-deployment configuration
# ---------------------------------------------------------------------------
# Expected .env variables:
#
# AZURE_OPENAI_API_KEY=<key>
# AZURE_OPENAI_FAST_DEPLOYMENT_URL=<complete working chat-completions URL>
#
# The URL is sent EXACTLY as configured. It does not need to contain an
# api-version query parameter when your enterprise Azure gateway does not use
# one.

def _find_env_file(filename=".env"):
    """Find .env in the current directory or one of its parents."""
    current = Path.cwd().resolve()

    for folder in (current, *current.parents):
        candidate = folder / filename
        if candidate.exists():
            return candidate

    return None


def _load_env_file(path):
    """
    Load .env values and overwrite stale values already held by the notebook
    kernel. This avoids having to restart the kernel after editing .env.
    """
    if path is None:
        return

    with path.open("r", encoding="utf-8") as env_file:
        for raw_line in env_file:
            line = raw_line.strip()

            if (
                not line
                or line.startswith("#")
                or "=" not in line
            ):
                continue

            key, value = line.split("=", 1)
            key = key.strip()
            value = (
                value.strip()
                .strip('"')
                .strip("'")
            )

            if key:
                os.environ[key] = value


def _clean_url(value):
    """
    Preserve the configured full URL while removing accidental quotes or
    markdown formatting copied from a portal or document.
    """
    if not value:
        return None

    value = (
        str(value)
        .strip()
        .strip('"')
        .strip("'")
        .strip()
    )

    markdown_match = re.search(
        r"\]\((https://[^)\s]+)\)",
        value,
    )
    if markdown_match:
        value = markdown_match.group(1).strip()

    https_positions = [
        match.start()
        for match in re.finditer(
            r"https://",
            value,
        )
    ]
    if https_positions:
        value = value[
            https_positions[-1]:
        ]

    return (
        value
        .strip()
        .strip("[]")
        .strip()
        .rstrip(").,;")
    )


ENV_PATH = _find_env_file()
_load_env_file(ENV_PATH)

AZURE_URL = _clean_url(
    os.getenv(
        "AZURE_OPENAI_FAST_DEPLOYMENT_URL"
    )
    or os.getenv(
        "AZURE_OPENAI_URL"
    )
    or os.getenv(
        "AZURE_OPENAI_CHAT_URL"
    )
    or os.getenv(
        "OPENAI_URL"
    )
)

AZURE_KEY = (
    os.getenv("AZURE_OPENAI_API_KEY")
    or os.getenv("AZURE_OPENAI_KEY")
    or os.getenv("OPENAI_API_KEY")
    or os.getenv("API_KEY")
)


def _validate_azure_configuration():
    if MOCK_MODE:
        return

    if not AZURE_URL:
        raise RuntimeError(
            "AZURE_OPENAI_FAST_DEPLOYMENT_URL is missing. "
            "Add the complete working chat-completions URL to .env."
        )

    if not AZURE_KEY:
        raise RuntimeError(
            "AZURE_OPENAI_API_KEY is missing from .env."
        )

    if not AZURE_URL.startswith("https://"):
        raise RuntimeError(
            "AZURE_OPENAI_FAST_DEPLOYMENT_URL must be a full HTTPS URL."
        )

    if "/chat/completions" not in AZURE_URL:
        raise RuntimeError(
            "AZURE_OPENAI_FAST_DEPLOYMENT_URL must include "
            "/chat/completions. No api-version query parameter is required."
        )

    if requests is None:
        raise ImportError(
            "The requests package is required. "
            "Install it with: pip install requests"
        )


_validate_azure_configuration()

print(
    "env file:",
    str(ENV_PATH)
    if ENV_PATH
    else "(not found)",
)
print(
    "endpoint:",
    (
        AZURE_URL[:80] + "..."
        if AZURE_URL
        else "(mock)"
    ),
)
print(
    "key:",
    "set"
    if AZURE_KEY
    else "none",
    "| MOCK_MODE:",
    MOCK_MODE,
)


In [ ]:
def azure_chat(
    messages,
    temperature=TEMPERATURE,
    max_tokens=MAX_TOKENS,
    json_mode=JSON_MODE,
    retries=4,
    timeout=120,
):
    """
    Call AZURE_OPENAI_FAST_DEPLOYMENT_URL exactly as configured.

    The endpoint may be a standard Azure URL or an enterprise gateway URL and
    does not need an api-version query parameter.

    Compatibility behavior:
    - sends the key in the Azure ``api-key`` header;
    - tries ``max_completion_tokens`` first;
    - falls back to ``max_tokens``;
    - retries transient HTTP and network failures;
    - retries without temperature if unsupported;
    - retries without response_format if JSON mode is unsupported.
    """
    if MOCK_MODE:
        return _mock_chat(messages)

    if requests is None:
        raise ImportError(
            "The requests package is required. "
            "Install it with: pip install requests"
        )

    headers = {
        "api-key": AZURE_KEY,
        "Content-Type": "application/json",
        "Accept": "application/json",
    }

    last_error = None

    for token_field in (
        "max_completion_tokens",
        "max_tokens",
    ):
        include_temperature = (
            temperature is not None
        )
        include_json_mode = bool(
            json_mode
        )

        while True:
            compatibility_retry = False

            for attempt in range(
                retries
            ):
                body = {
                    "messages": messages,
                    token_field: max_tokens,
                }

                if include_temperature:
                    body["temperature"] = (
                        temperature
                    )

                if include_json_mode:
                    body["response_format"] = {
                        "type": "json_object"
                    }

                try:
                    response = requests.post(
                        AZURE_URL,
                        headers=headers,
                        json=body,
                        timeout=timeout,
                    )

                except requests.RequestException as exc:
                    last_error = exc

                    if attempt < retries - 1:
                        wait = min(
                            2 ** attempt,
                            10,
                        )
                        print(
                            "Azure connection error; "
                            f"retrying in {wait}s "
                            f"({attempt + 1}/{retries})"
                        )
                        time.sleep(wait)
                        continue

                    break

                if response.status_code == 200:
                    try:
                        payload = response.json()
                        return payload[
                            "choices"
                        ][0]["message"]["content"]
                    except (
                        ValueError,
                        KeyError,
                        IndexError,
                        TypeError,
                    ) as exc:
                        raise RuntimeError(
                            "Azure returned an unexpected "
                            "successful response: "
                            f"{response.text[:1000]}"
                        ) from exc

                error_text = response.text
                error_lower = (
                    error_text.lower()
                )

                if (
                    include_temperature
                    and response.status_code
                    in (400, 422)
                    and "temperature"
                    in error_lower
                ):
                    include_temperature = False
                    compatibility_retry = True
                    print(
                        "Azure rejected temperature; "
                        "retrying without it."
                    )
                    break

                if (
                    include_json_mode
                    and response.status_code
                    in (400, 422)
                    and (
                        "response_format"
                        in error_lower
                        or "json_object"
                        in error_lower
                    )
                ):
                    include_json_mode = False
                    compatibility_retry = True
                    print(
                        "Azure rejected response_format; "
                        "retrying without JSON mode."
                    )
                    break

                if (
                    token_field
                    == "max_completion_tokens"
                    and response.status_code
                    in (400, 422, 500)
                    and (
                        "max_completion_tokens"
                        in error_lower
                        or "unsupported"
                        in error_lower
                        or "invalid parameter"
                        in error_lower
                    )
                ):
                    last_error = RuntimeError(
                        "Azure rejected "
                        "max_completion_tokens: "
                        f"{error_text[:600]}"
                    )
                    print(
                        "Azure rejected "
                        "max_completion_tokens; "
                        "trying max_tokens."
                    )
                    break

                if response.status_code in (
                    429,
                    500,
                    502,
                    503,
                    504,
                ):
                    last_error = RuntimeError(
                        "Azure HTTP "
                        f"{response.status_code}: "
                        f"{error_text[:600]}"
                    )

                    if attempt < retries - 1:
                        retry_after = (
                            response.headers.get(
                                "Retry-After"
                            )
                        )

                        try:
                            wait = (
                                float(retry_after)
                                if retry_after
                                is not None
                                else min(
                                    2 ** attempt,
                                    10,
                                )
                            )
                        except ValueError:
                            wait = min(
                                2 ** attempt,
                                10,
                            )

                        print(
                            "Azure HTTP "
                            f"{response.status_code}; "
                            f"retrying in {wait}s "
                            f"({attempt + 1}/{retries})"
                        )
                        time.sleep(wait)
                        continue

                    break

                raise RuntimeError(
                    "Azure request failed with "
                    f"HTTP {response.status_code}: "
                    f"{error_text[:1200]}"
                )

            if compatibility_retry:
                continue

            break

    raise RuntimeError(
        "Azure request failed after all retries "
        "and compatibility fallbacks. "
        f"Last error: {last_error}"
    )


def parse_contract(raw):
    s = raw.strip()

    if s.startswith("```"):
        s = re.sub(
            r"^```[a-zA-Z]*\n?",
            "",
            s,
        )
        s = re.sub(
            r"\n?```$",
            "",
            s,
        )

    return json.loads(s)


In [ ]:
# Mock Azure client: parses the labelled evidence lines and cites real ids/values so the gate passes.
def _mock_chat(messages):
    user = messages[1]["content"] if len(messages) > 1 else messages[0]["content"]
    mode = ("narrative" if "MODE: narrative" in user else "absence" if "MODE: absence" in user else "data_backed")
    reqids = re.findall(r"\[([A-Z0-9_]+)\]", user.split("EVIDENCE")[0])
    # evidence lines look like: "- E-BANK01-0037: Scope 1 emissions = 2,634.9 tCO2e [measure, 2024] -- ..."
    ev = re.findall(r"- (E-BANK\d+-\d+): [^=\n]*= ([\-]?\d[\d,]*\.?\d*)\s*([A-Za-z%\u20ac/]*)", user)
    if mode == "narrative" or not ev:
        return json.dumps({"prose": "The entity confirms compliance with the applicable disclosure requirements.",
            "citations_used": [], "numeric_claims": [], "requirements_addressed": reqids[:6],
            "standards_covered": ["IFRS S1"]})
    picks = ev[:3]; cids, claims, frags = [], [], []
    for cid, num, unit in picks:
        cids.append(cid); claim = f"{num} {unit}".strip()
        claims.append({"text": claim, "citation_id": cid})
        frags.append(f"{claim} [{cid}]")
    return json.dumps({"prose": "For the reporting period, the disclosed figures were " + "; ".join(frags) + ".",
        "citations_used": cids, "numeric_claims": claims,
        "requirements_addressed": reqids[:8], "standards_covered": ["IFRS S1", "IFRS S2"]})
print("mock client (labelled-format aware) defined")

In [ ]:
blocks = json.loads((OUT / f"generation_blocks_{BANK}.json").read_text(encoding="utf-8"))
store  = json.loads((OUT / f"evidence_store_{BANK}.json").read_text(encoding="utf-8"))
plan   = json.loads((OUT / f"section_plan_{BANK}.json").read_text(encoding="utf-8"))
blocks_by_id = {b["block_id"]: b for b in blocks}
SECTION_TITLE = {s["section_key"]: s["section_title"] for s in plan["sections"]}

def deref(path):
    cur = store
    for p in path.strip("/").split("/"):
        cur = cur[p.replace("~1", "/").replace("~0", "~")]
    return cur
print(f"{len(blocks)} blocks | {sum(len(s['subsections']) for s in plan['sections'])} subsections | store {len(store)} collections")

In [ ]:
# ================= ANTI-HALLUCINATION HARDENING LAYER =================
import re

# ---- (6) evidence labelling: give every value a human meaning so it can't be misread ----
FIELD_DICT = {
 "scope1_total_tco2e": ("Scope 1 emissions", "Total direct GHG emissions"),
 "scope1_gas_tco2e": ("Scope 1 \u2013 stationary/gas", "Scope 1 from gas/stationary combustion"),
 "scope1_fleet_tco2e": ("Scope 1 \u2013 fleet", "Scope 1 from owned vehicles"),
 "scope2_market_tco2e": ("Scope 2 (market-based)", "Market-based Scope 2 emissions"),
 "scope2_location_tco2e": ("Scope 2 (location-based)", "Location-based Scope 2 emissions"),
 "scope3_travel_tco2e": ("Scope 3 \u2013 business travel", "Business-travel Scope 3 emissions"),
 "emissions_tco2e": ("Scope 3 category emissions", "Emissions for a Scope 3 category"),
 "financed_emissions_tco2e": ("Financed emissions", "PCAF financed emissions"),
 "financed_emissions_2024_tco2e": ("Financed emissions (2024)", "Total financed emissions"),
 "carbon_intensity_tco2e_per_meur": ("Financed-emissions intensity", "tCO2e per \u20ac million"),
 "attribution_factor": ("Attribution factor", "PCAF attribution share (0\u20131)"),
 "pcaf_data_quality_score": ("PCAF data-quality score", "PCAF score 1(best)\u20135(worst)"),
 "audited_report": ("Audited-data contribution", "PCAF weight of audited-report data (NOT an assurance level)"),
 "assurance_scope": ("Assurance scope", "Scope of external assurance obtained"),
 "risk_rating": ("Risk rating", "5\u00d75 matrix rating"),
 "likelihood_score": ("Likelihood score", "Risk likelihood (1\u20135)"),
 "severity_score": ("Severity score", "Risk severity (1\u20135)"),
 "board_climate_expertise_pct": ("Board climate expertise", "% of board with climate expertise"),
 "ceo_esg_compensation_pct": ("CEO ESG-linked pay", "% of CEO pay linked to ESG"),
 "esg_committee_meetings": ("ESG committee meetings", "Number of ESG committee meetings"),
 "target_year": ("Target year", None), "baseline_year": ("Baseline year", None),
 "schedule_elapsed_pct": ("Schedule elapsed", "% of target period elapsed"),
 "progress_basis": ("Progress basis", "How progress is measured"),
 "consolidation_approach": ("Consolidation approach", "Emissions consolidation boundary"),
}
def _humanize_field(f):
    return re.sub(r"_tco2e$"," (tCO2e)",str(f)).replace("_pct"," (%)").replace("_"," ").strip().capitalize()
def semantic_type(value):
    if isinstance(value, bool): return "flag"
    if isinstance(value, (int, float)): return "measure"
    s = str(value)
    if re.fullmatch(r"[a-z0-9]+(?:_[a-z0-9]+)+", s): return "enum"
    if " " not in s and re.fullmatch(r"[A-Z0-9][A-Za-z0-9\-_/.]{1,19}", s): return "code"
    if len(s.split()) >= 6: return "text"
    return "label"
def humanize_value(v):
    if isinstance(v, bool): return "Yes" if v else "No"
    if isinstance(v, float):
        a = abs(v)
        if a >= 1e6: return f"{v/1e6:.2f} million"
        if a >= 1000: return f"{v:,.1f}"
        return (f"{v:,.4f}").rstrip("0").rstrip(".")
    if isinstance(v, int): return f"{v:,}"
    s = str(v)
    if re.fullmatch(r"[a-z0-9]+(?:_[a-z0-9]+)+", s): return s.replace("_", " ").capitalize()
    return s
def label_evidence(blocks):
    for b in blocks:
        for e in b["evidence"]:
            lbl, desc = FIELD_DICT.get(e["field"], (_humanize_field(e["field"]), None))
            e["label"] = lbl; e["description"] = desc
            e["semantic_type"] = semantic_type(e["value"])
            e["display_value"] = humanize_value(e["value"])
label_evidence(blocks)

# ---- (hygiene) citation validity: only E-BANK.... ids are real citations ----
E_CITE = re.compile(r"E-BANK\d+-\d{4,}")
def strip_bad_citations(out, prose_key="prose"):
    stripped = []
    # drop non-E ids from the contract
    cu = [c for c in out.get("citations_used", []) if E_CITE.fullmatch(c)]
    stripped += [c for c in out.get("citations_used", []) if not E_CITE.fullmatch(c)]
    out["citations_used"] = cu
    out["numeric_claims"] = [nc for nc in out.get("numeric_claims", []) if E_CITE.fullmatch(str(nc.get("citation_id","")))]
    # remove any [IFRS_...] or other non-E bracket citations from the prose text
    if prose_key in out and isinstance(out[prose_key], str):
        out[prose_key] = re.sub(r"\[(?!E-BANK)[A-Za-z0-9_\-]+\]", "", out[prose_key])
        out[prose_key] = re.sub(r"[ \t]{2,}", " ", out[prose_key])
    return out, stripped

# ---- (2) denylist scanner: fabrication classes with no payload source ----
DENYLIST = [r"\bappendix\b", r"\bhyperlink\b", r"\bportal\b", r"https?://", r"\bwww\.",
            r"technical appendix", r"assurance report", r"methodology note",
            r"\brestat(?:ed|ement)\b", r"subsequent event", r"prior[- ]period error",
            r"\bunreserved\b", r"comply with all", r"in accordance with all applicable",
            r"available (?:at|via|through)[^.\n]{0,60}(?:report|portal|website|hyperlink)"]
DENY_RE = [re.compile(p, re.I) for p in DENYLIST]
def scan_denylist(prose):
    hits = []
    for rx in DENY_RE:
        m = rx.search(prose or "")
        if m: hits.append(m.group(0))
    return hits

# ---- (3) uncited-claim detector: a non-year number with no citation in its sentence ----
def scan_uncited(prose):
    """Flag sentences that state a genuine quantitative MEASURE without a citation.
    Ignores ordinals/labels (Scope 1/2/3, Category 6, (i)/(ii)), years, and citation-id fragments."""
    bad = []
    for sent in re.split(r"(?<=[.;:])\s+", prose or ""):
        if "[E-BANK" in sent:                      # sentence carries a citation -> fine
            continue
        s = re.sub(r"\[[^\]]*\]", "", sent)         # drop any bracketed refs before scanning
        measures = re.findall(
            r"-?\d[\d,]*\.\d+"                        # decimals: 2,634.9
            r"|\d{1,3}(?:,\d{3})+"                    # thousands: 35,973,167
            r"|\d+(?:\.\d+)?\s?(?:tCO2e|tco2e|%|MEUR|EUR|\u20ac|tonnes|tonne)",  # unit-attached
            s)
        for n in re.findall(r"\b\d+\b", s):          # large bare integers (skip ordinals/years)
            if 31 < int(n) and not re.fullmatch(r"(?:19|20)\d\d", n):
                measures.append(n)
        if measures:
            bad.append(sent.strip()[:80])
    return bad[:5]

print("hardening layer loaded: labels + citation-strip + denylist + uncited scanners")


In [ ]:
# ---- (1) high-risk sentences are DETERMINISTIC, never LLM-generated ----
def _mandatory_complete(blocks):
    # complete only if nothing is a gap/narrative/absence anywhere
    for b in blocks:
        if b["generation_mode"] in ("absence", "narrative") or b.get("data_gaps"):
            return False
    return True

def compliance_statement(blocks):
    if _mandatory_complete(blocks):
        return ("**Statement of compliance.** For the year ended 31 December 2024, the Bank\u2019s "
                "sustainability-related financial disclosures comply with IFRS S1 and IFRS S2.")
    return ("**Basis of preparation.** This report has been prepared with reference to, and is intended to be "
            "aligned with, IFRS S1 and IFRS S2. It does not constitute a statement of full compliance, "
            "as some required information is unavailable or subject to measurement uncertainty (see the relevant "
            "sections and stated data gaps).")

def assurance_statement(blocks):
    scope = None
    for b in blocks:
        for e in b["evidence"]:
            if e["field"] == "assurance_scope":
                scope = humanize_value(e["value"]); break
        if scope: break
    if scope:
        return f"**Assurance.** Limited assurance was obtained over {scope} under ISAE 3000. Other metrics and targets in this report are not covered by external assurance."
    return "**Assurance.** No external assurance was obtained over the disclosures in this report."

# ---- (5) deterministic tables from evidence (kills raw enums/decimals + shrinks prose) ----
TABLE_SUBSECTIONS = {"Greenhouse Gas Emissions", "Financed Emissions", "Climate-Related Targets",
                     "Climate Resilience & Scenario Analysis", "Climate-Related Metrics"}
def subsection_table(sub_blocks):
    rows, seen = [], set()
    for b in sub_blocks:
        for e in b["evidence"]:
            if e["semantic_type"] not in ("measure", "flag"): continue
            key = (e["label"], e.get("reporting_year"))
            if key in seen: continue
            seen.add(key)
            unit = e["unit"] or ""
            rows.append((e["label"], e["display_value"], unit, str(e.get("reporting_year") or ""), e["citation_id"]))
    if len(rows) < 3: return ""
    rows.sort(key=lambda r: r[0])
    out = ["", "| Metric | Value | Unit | Year | Ref |", "| --- | ---: | --- | ---: | --- |"]
    out += [f"| {a} | {b} | {c} | {d} | [{e}] |" for a, b, c, d, e in rows[:60]]
    return "\n".join(out) + "\n"
print("templates + table builder loaded")


In [ ]:
STYLE_GUIDE = ("STYLE: formal regulatory disclosure prose; concise; no marketing language; "
               "integrate IFRS S1 and S2 into one narrative where both apply.")  # inject your V9.7 style here

MODE_INSTRUCTION = {
 "data_backed": ("Write the disclosure strictly from the evidence. Every factual sentence must end with its "
                 "citation(s) in square brackets, e.g. [E-BANK01-0037]. State no number or fact not in the evidence."),
 "absence":     ("The required data is unavailable. State the absence following the gap instruction verbatim. "
                 "Do NOT report missing values as zero and do NOT invent figures."),
 "narrative":   ("No quantitative data applies. Write a brief qualitative compliance statement addressing the "
                 "requirement(s). Use NO numbers and cite no evidence."),
}

GUARDRAILS = (
 "STRICT RULES (a violation makes the output invalid):\n"
 "1. The EVIDENCE list is the complete and only set of facts. Use no outside knowledge.\n"
 "2. Do NOT mention, name, cite, or link to any external document, appendix, report, note, portal or URL.\n"
 "3. Do NOT use 'restatement', 'prior-period error', 'subsequent event', or unreserved-compliance language.\n"
 "4. Do NOT write any statement of compliance or of assurance -- these are added separately by the system.\n"
 "5. Address ONLY the requirements listed below; introduce no other topic.\n"
 "6. Every sentence stating a number or fact must carry a citation [E-...]. If a required item has no evidence, "
 "write one sentence saying it is not disclosed -- do not improvise.\n"
 "7. Use the evidence 'label' meanings; never reinterpret a value beyond its stated meaning."
)

def build_messages(block, section_title):
    reqs = "\n".join(
        f"- [{r['requirement_id']}] ({r['standard']} \u00b6{r['paragraph_id']}{r['clause_path'] or ''}): {r['requirement_text']}"
        for r in block["requirements"])
    def ev_line(e):
        d = f" -- {e['description']}" if e.get("description") else ""
        return (f"- {e['citation_id']}: {e.get('label', e['field'])} = {e.get('display_value', e['value'])} "
                f"{e['unit'] or ''} [{e.get('semantic_type','')}, {e.get('reporting_year')}]{d}")
    evidence = "\n".join(ev_line(e) for e in block["evidence"]) or "(no evidence -- state not disclosed)"
    gaps = "\n".join(f"- {g['field']}: {g.get('instruction','')}" for g in block.get("data_gaps", []))
    system = ("You are an expert sustainability-reporting writer preparing a bank's IFRS S1 and IFRS S2 aligned "
              "climate-related disclosure. You write only what the requirements ask and only what the evidence "
              "supports. You never invent figures, documents, or assurance/compliance claims. You cite every fact.")
    user = f"""SECTION: {section_title} > {block['subsection_title']}
CONCEPT: {block['concept']}    STANDARDS: {', '.join(block['standards_present'])}    MODE: {block['generation_mode']}

{GUARDRAILS}

REQUIREMENTS (produce ONE aligned disclosure satisfying all; integrate S1 and S2 where both apply):
{reqs}

EVIDENCE (the ONLY facts you may state; cite by citation_id):
{evidence}
{('GAPS:' + chr(10) + gaps) if gaps else ''}
INSTRUCTION: {MODE_INSTRUCTION[block['generation_mode']]}
{STYLE_GUIDE}

Return ONLY a JSON object:
{{"prose": "disclosure text with a [E-...] citation after every figure/fact",
  "citations_used": ["E-..."],
  "numeric_claims": [{{"text": "<figure and unit only, e.g. 2,634.9 tCO2e>", "citation_id": "E-..."}}],
  "requirements_addressed": ["<requirement_id>"],
  "standards_covered": ["IFRS S1", "IFRS S2"]}}"""
    return [{"role": "system", "content": system}, {"role": "user", "content": user}]

In [ ]:
def _numbers(s):
    """All numbers in a string, scale-word aware ('35.97 million' -> 3.597e7).
    Only word scales (million/billion/thousand/bn) count, so 'MEUR'/'km' do not collide."""
    out = []
    for m in re.finditer(r"(-?\d[\d,]*\.?\d*)\s*(million|billion|thousand|bn)?", str(s), re.I):
        if not m.group(1):
            continue
        n = float(m.group(1).replace(",", ""))
        mult = {"thousand": 1e3, "million": 1e6, "billion": 1e9, "bn": 1e9}.get((m.group(2) or "").lower(), 1)
        out.append(n * mult)
    return out

def _to_number(s):
    ns = _numbers(s)
    return ns[0] if ns else None

def _leaf_numbers(v):
    if isinstance(v, dict):
        out = []
        for x in v.values(): out += _leaf_numbers(x)
        return out
    if isinstance(v, (int, float)) and not isinstance(v, bool):
        return [float(v)]
    return []

def _match(claim_nums, target, tol_frac=0.005):
    return any(abs(n - target) <= max(abs(target) * tol_frac, 0.01) for n in claim_nums)

def gate(block, out):
    """Re-resolve every citation to the store; reject invented ids and mismatched numbers.
    The stored value must appear among the numbers in the claim text (robust to leading labels
    like 'Scope 1' or a trailing year). Text/qualitative evidence is not numeric-gated."""
    ev = {e["citation_id"]: e for e in block["evidence"]}
    fails = []
    for c in out.get("citations_used", []):
        if c not in ev:
            fails.append(f"cites unknown citation_id {c}"); continue
        if deref(ev[c]["path"]) != ev[c]["value"]:
            fails.append(f"evidence pointer drift for {c}")
    for nc in out.get("numeric_claims", []):
        cid = nc.get("citation_id")
        if cid not in ev:
            fails.append(f"numeric claim cites unknown id {cid}"); continue
        stored = ev[cid]["value"]
        claim_nums = _numbers(nc.get("text"))
        if not claim_nums:
            continue
        if isinstance(stored, bool):
            fails.append(f"numeric claim on boolean evidence {cid}")
        elif isinstance(stored, (int, float)):
            if not _match(claim_nums, stored):
                fails.append(f"number '{nc.get('text')}' != evidence {stored} for {cid}")
        elif isinstance(stored, dict):
            leaves = _leaf_numbers(stored)
            if leaves and not any(_match(claim_nums, l) for l in leaves):
                fails.append(f"number '{nc.get('text')}' matches no value in {cid}")
        # str / list stored values: not numeric-gated
    if block["generation_mode"] == "narrative" and out.get("numeric_claims"):
        fails.append("narrative block must contain no numeric claims")
    if block["generation_mode"] == "data_backed" and block["evidence"] and not out.get("citations_used"):
        fails.append("data_backed block produced no citations")
    ds = scan_denylist(out.get("prose", ""))
    if ds: fails.append("forbidden content: " + ", ".join(sorted(set(ds))[:4]))
    uc = scan_uncited(out.get("prose", ""))
    if uc: fails.append("uncited numeric sentence(s): " + " | ".join(uc[:3]))
    return (len(fails) == 0, fails)

In [ ]:
def generate_block(block, section_title, max_revisions=MAX_REVISIONS):
    messages = build_messages(block, section_title)
    last_out, last_fails = None, ["no output"]
    for attempt in range(max_revisions + 1):
        raw = azure_chat(messages)
        try:
            out = parse_contract(raw)
        except Exception as e:
            messages += [{"role": "assistant", "content": raw},
                         {"role": "user", "content": f"That was not valid JSON ({e}). Return ONLY the JSON object."}]
            last_fails = [f"invalid JSON: {e}"]; continue
        out, _stripped = strip_bad_citations(out)
        ok, fails = gate(block, out)
        if ok:
            return {"ok": True, "attempts": attempt, "output": out, "failures": []}
        last_out, last_fails = out, fails
        messages += [{"role": "assistant", "content": raw},
                     {"role": "user", "content": "Fix ALL of these and return corrected JSON only:\n- " + "\n- ".join(fails)}]
    return {"ok": False, "attempts": max_revisions, "output": last_out, "failures": last_fails}

In [ ]:
# ================= TIER 2: SECTION / SUBSECTION EDITOR =================
# Merges the per-block drafts of a subsection into ONE coherent, deduplicated narrative.
# Gated so the editor can only REUSE existing citation ids and numbers -- it cannot add facts.
EDITOR_ENABLED    = True
EDITOR_MIN_BLOCKS = 1      # edit subsections with >= this many drafts (1 = tighten everything)
EDITOR_MAX_TOKENS = 8000
EDITOR_MAX_REVISIONS = 2

def _subsection_evidence(sub_blocks):
    ev = {}
    for b in sub_blocks:
        for e in b["evidence"]:
            ev[e["citation_id"]] = e
    return ev

def build_editor_messages(section_title, subsection_title, drafts, ev_map):
    allowed = "\n".join(
        f"- {cid}: {e.get('label', e['field'])} = {e.get('display_value', e['value'])} {e['unit'] or ''}"
        for cid, e in ev_map.items())
    drafts_txt = "\n\n--- DRAFT ---\n\n".join(drafts)
    system = ("You are a regulatory-disclosure editor. You MERGE draft passages into one coherent, "
              "non-repetitive subsection. You never add facts, numbers, documents, or citations; you only "
              "reuse what the drafts already contain, and you keep every figure attached to its citation.")
    user = f"""SECTION: {section_title} > {subsection_title}

Merge the DRAFTS below into a SINGLE coherent, concise disclosure for this subsection.
RULES:
- Remove all repetition; state each fact and each figure once.
- Keep only content relevant to "{subsection_title}"; drop anything that belongs to another topic.
- You may use ONLY the citation ids in ALLOWED CITATIONS. Keep each figure immediately followed by its [E-...].
- Introduce NO new number, fact, document, URL, or claim. Write NO compliance or assurance statement.
- Aim for roughly half the combined length: tight regulatory prose.

ALLOWED CITATIONS (the only ids/values you may use):
{allowed}

DRAFTS:
{drafts_txt}

Return ONLY JSON: {{"prose": "merged disclosure with [E-...] after every figure",
  "citations_used": ["E-..."],
  "numeric_claims": [{{"text": "<figure and unit>", "citation_id": "E-..."}}]}}"""
    return [{"role": "system", "content": system}, {"role": "user", "content": user}]

def edit_subsection(section_title, ss, generated, blocks_by_id):
    sub_blocks = [blocks_by_id[b["block_id"]] for b in ss["blocks"] if b["block_id"] in blocks_by_id]
    drafts = [generated[b["block_id"]]["output"]["prose"]
              for b in ss["blocks"]
              if generated.get(b["block_id"]) and generated[b["block_id"]].get("output")]
    drafts = [d for d in drafts if d and d.strip()]
    if not drafts:
        return {"ok": True, "prose": "", "attempts": 0, "failures": [], "edited": False}
    if not EDITOR_ENABLED or len(drafts) < EDITOR_MIN_BLOCKS:
        return {"ok": True, "prose": "\n\n".join(drafts), "attempts": 0, "failures": [], "edited": False}

    ev_map = _subsection_evidence(sub_blocks)
    synth = {"evidence": list(ev_map.values()), "generation_mode": "data_backed", "data_gaps": []}
    messages = build_editor_messages(section_title, ss["subsection_title"], drafts, ev_map)
    fails = ["no output"]
    for attempt in range(EDITOR_MAX_REVISIONS + 1):
        raw = azure_chat(messages, max_tokens=EDITOR_MAX_TOKENS)
        try:
            out = parse_contract(raw)
        except Exception as e:
            messages += [{"role": "assistant", "content": raw},
                         {"role": "user", "content": f"That was not valid JSON ({e}). Return ONLY the JSON object."}]
            fails = [f"invalid JSON: {e}"]; continue
        out, _ = strip_bad_citations(out)
        ok, fails = gate(synth, out)
        # extra: inline [E-...] in prose must all belong to this subsection
        inline = set(re.findall(r"\[(E-BANK\d+-\d+)\]", out.get("prose", "")))
        bad_inline = [c for c in inline if c not in ev_map]
        if bad_inline:
            ok = False; fails = fails + ["inline citation(s) not in this subsection: " + ", ".join(bad_inline[:3])]
        if ok:
            return {"ok": True, "prose": out["prose"], "attempts": attempt, "failures": [], "edited": True}
        messages += [{"role": "assistant", "content": raw},
                     {"role": "user", "content": "Fix ALL of these and return corrected JSON only:\n- " + "\n- ".join(fails)}]
    # never lose content: fall back to the concatenated (already-gated) drafts
    return {"ok": False, "prose": "\n\n".join(drafts), "attempts": EDITOR_MAX_REVISIONS, "failures": fails, "edited": False}

def run_editor(plan, generated, blocks_by_id):
    edited, stats = {}, {"edited": 0, "passthrough": 0, "failed": 0}
    for s in plan["sections"]:
        for ss in s["subsections"]:
            res = edit_subsection(s["section_title"], ss, generated, blocks_by_id)
            edited[(s["section_key"], ss["subsection_order"])] = res
            stats["failed" if not res["ok"] else ("edited" if res["edited"] else "passthrough")] += 1
            print(f"  edit {s['section_key']}.{ss['subsection_order']} {ss['subsection_title'][:32]:32s} "
                  f"edited={res['edited']} ok={res['ok']} att={res['attempts']}")
    return edited, stats
print("Tier 2 section editor loaded")


In [ ]:
def run_writer(plan, blocks_by_id):
    generated, report_stats = {}, {"ok": 0, "failed": 0, "attempts": 0}
    for s in plan["sections"]:
        for ss in s["subsections"]:
            for bref in ss["blocks"]:
                block = blocks_by_id[bref["block_id"]]
                res = generate_block(block, s["section_title"])
                generated[bref["block_id"]] = res
                report_stats["attempts"] += res["attempts"]
                report_stats["ok" if res["ok"] else "failed"] += 1
                flag = "" if res["ok"] else f"  !! {res['failures']}"
                print(f"  {bref['block_id']:44s} attempts={res['attempts']} ok={res['ok']}{flag}")
    return generated, report_stats

def assemble(plan, generated, edited=None):
    comp = compliance_statement(blocks)
    assur = assurance_statement(blocks)
    L = [f"# IFRS S1 & S2 Aligned Climate-Related Disclosure — {plan['bank_id']}", ""]
    for s in plan["sections"]:
        L += [f"## {s['order']}. {s['section_title']}", ""]
        if s["section_key"] == "general_requirements":
            L += [comp, ""]
        for ss in s["subsections"]:
            L += [f"### {s['order']}.{ss['subsection_order']} {ss['subsection_title']}", ""]
            sub_blocks = [blocks_by_id[b["block_id"]] for b in ss["blocks"] if b["block_id"] in blocks_by_id]
            ed = edited.get((s["section_key"], ss["subsection_order"])) if edited else None
            if ed and ed.get("prose"):
                L += [ed["prose"].strip(), ""]
            else:
                for bref in ss["blocks"]:
                    res = generated.get(bref["block_id"])
                    if res and res["output"]:
                        L += [res["output"]["prose"].strip(), ""]
            if ss["subsection_title"] in TABLE_SUBSECTIONS:
                tbl = subsection_table(sub_blocks)
                if tbl: L += [tbl]
        if s["section_key"] == "metrics_and_targets":
            L += [assur, ""]
    return "\n".join(L)

generated, stats = run_writer(plan, blocks_by_id)
print("\n-- Tier 2 section editor --")
edited, edit_stats = run_editor(plan, generated, blocks_by_id)
print("editor:", edit_stats)
print("\nblocks:", stats)
report_md = assemble(plan, generated, edited)
(OUT / f"report_{BANK}.md").write_text(report_md, encoding="utf-8")
# audit sidecar: every citation used across the report, resolvable to a pointer
audit = [{"block_id": bid, "ok": r["ok"], "attempts": r["attempts"],
          "citations": (r["output"] or {}).get("citations_used", []), "failures": r["failures"]}
         for bid, r in generated.items()]
(OUT / f"report_{BANK}_audit.json").write_text(json.dumps(audit, ensure_ascii=False, indent=2), encoding="utf-8")
print(f"wrote {OUT}/report_{BANK}.md  ({len(report_md)} chars) + audit sidecar")